In [7]:
import sys

sys.path.append("..")

import pandas as pd 
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer 
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from src.data.cleaning import (
    clean_dataset
)

from src.features.engineering import (
    create_features
)

In [3]:
DATA_PATH='../data/raw/car details v4.csv'

df=pd.read_csv(DATA_PATH)
df=clean_dataset(df)
df=df.drop(columns=['torque','engine','max_power'])


## Seprating Target


In [4]:
X=df.drop(columns='selling_price')
y=df['selling_price']

## Train Test Split

In [5]:
X_train,X_test,y_train,y_test=train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
    )

## Creating features

In [8]:
X_train=create_features(X_train)
X_test=create_features(X_test)

## Identifying Numerical and Categorical Fetaures

In [9]:
numeric_features= X_train.select_dtypes(
    include="number"
).columns.tolist()
categorical_features= X_train.select_dtypes(
    exclude="number"
).columns.tolist()


print("Numeric:", numeric_features)
print("Categorical:", categorical_features)

Numeric: ['year', 'km_driven', 'mileage', 'seats', 'engine_cc', 'max_power_bhp', 'torque_nm', 'torque_rpm_min', 'torque_rpm_max', 'age']
Categorical: ['fuel', 'seller_type', 'transmission', 'owner', 'brand']


In [11]:
numeric_pipeline=SimpleImputer(
    strategy='median'
)

In [12]:
categorical_pipeline=Pipeline([
    (
        'imputer',
        SimpleImputer(strategy='most_frequent')
    ),
    (
        'encoder',
        OneHotEncoder(
            handle_unknown='ignore',
            sparse_output=True
        )
    )
])

In [13]:
preprocessor=ColumnTransformer(
    transformers=[
        ('numeric',
        numeric_pipeline,
        numeric_features),
        (
            'categorical',
            categorical_pipeline,
            categorical_features
        )
    ]
)

## Handling Missing Values

In [14]:
X_train_processed=preprocessor.fit_transform(X_train)

X_test_processed=preprocessor.transform(X_test)